# RQ1 matched — LR vs LR+ — Bảng A paper split + Bảng B 10 re-split

Hai biến thể dùng cùng cấu hình `{"C": 10.0, "class_weight": null}`, cùng split và cùng 5 seed `12–16`.
Khác biệt khái niệm duy nhất: `LR+` thêm 12 UFE label-free; `LR` chỉ dùng 14 đặc trưng gốc.

- **Bảng A:** split paper 1657/553/553, retrospective.
- **Bảng B:** 10 split phân tầng mới `101..1010`, đơn vị thống kê là split sau khi trung bình 5 model seed.
- Cấu hình đã được khóa từ validation trước đây; notebook này không tuning lại theo test.
- Bảng B không phải external holdout. PyG sampler có thể khác giữa phiên, nhưng hai biến thể trong mỗi split dùng chung batch/feature materialization.


In [1]:

from pathlib import Path
import csv
import gc
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

import numpy as np
import torch

MODEL_KEY = 'logistic_regression'
MODEL_LABEL = 'Logistic Regression'
VARIANTS = ["base", "plus"]
VARIANT_LABEL = {"base": 'LR', "plus": 'LR+'}
SHARED_CONFIG = {'C': 10.0, 'class_weight': None}
MODEL_SEEDS = [12, 13, 14, 15, 16]
RESPLIT_SEEDS = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010]
PAPER_SAMPLER_SEEDS = {"train": 1201, "validation": 1202, "test": 2301}
EXPECTED_COUNTS = {"train": 1657, "validation": 553, "test": 553}
SOURCE_DATA_SHA256 = "8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8"
EDGE_STATS_DIM = 12
RELATION_TO_INDEX = {"call": 0, "trans": 1}
THRESHOLD = 0.5
BOOTSTRAP_REPETITIONS = 20_000
MINIMUM_PRACTICAL_DELTA = 0.005
ALPHA = 0.05
CODE_SHA256 = "327f9119b367313e20739c28bed4b782af947df7e109051f9da634a3e6eec434"
CONFIG = {'protocol_id': 'group23-rq1-logistic_regression-matched-paper-and-resplits-v1', 'status': 'FROZEN AFTER PAPER-SPLIT EXPLORATION AND BEFORE 10-RESPLIT EVALUATION', 'dataset': 'HEIG-Phish author processed data.pt', 'source_data_sha256': '8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8', 'task': 'static transductive EOA binary node classification', 'model_pair': {'baseline': 'LR', 'plus': 'LR+'}, 'shared_configuration': {'C': 10.0, 'class_weight': None}, 'implementation': {'estimator': 'StandardScaler + sklearn LogisticRegression', 'solver': 'liblinear', 'max_iter': 5000}, 'configuration_origin': 'frozen baseline winner selected on the paper validation split', 'matching_rule': 'Both variants use the same fixed configuration, split, model seed, threshold, and sampled target batches. Plus adds only the 12-dimensional label-free UFE branch/input.', 'paper_split': {'counts': [1657, 553, 553], 'role': 'retrospective paper-aligned matched comparison', 'sampler_seeds': {'train': 1201, 'validation': 1202, 'test': 2301}}, 'resplits': {'method': 'stratified random 60/20/20 over the same 2763 labeled EOA', 'split_seeds': [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010], 'role': 'pre-registered re-split stability; not an external holdout', 'sampler_seeds': 'train/validation/test = split_seed*10 + 1/2/3'}, 'model_seeds': [12, 13, 14, 15, 16], 'primary_metric': 'Macro-F1 at threshold 0.5', 'secondary_metrics': ['PR-AUC', 'ROC-AUC', 'precision', 'recall', 'binary-F1', 'micro-F1', 'MCC', 'FPR', 'fit time', 'inference time'], 'ufe': {'dimensions': 12, 'definition': 'rows/count/sum x in/out x call/trans inside each sampled batch', 'transform': 'log1p', 'uses_labels': False, 'fanout': [100, 100], 'batch_size': 512, 'reproducibility_caveat': 'PyG sampling can differ between sessions; within each pair both variants reuse exactly the same materialized batches/features.'}, 'selection': {'hyperparameter_tuning_in_this_notebook': False, 'validation_role': 'checkpoint selection for GNNs and pre-test diagnostic for tabular models', 'threshold': 0.5, 'test_before_freeze': False}, 'inference': {'table_a': 'paired node bootstrap of mean per-seed Macro-F1; retrospective', 'table_b': 'Nadeau-Bengio corrected resampled t-test on 10 split-level deltas after averaging five model seeds', 'minimum_practical_delta': 0.005, 'alpha': 0.05, 'global_multiplicity': 'Apply Holm across all six model-pair Table-B p-values in the final RQ1 aggregation; notebook reports raw pair result.'}, 'code_sha256': '__CODE_SHA256__', 'change_control': 'Any change after viewing Table-B results requires a new protocol version and exploratory labeling.'}
CONFIG["code_sha256"] = CODE_SHA256
PROTOCOL_JSON = json.dumps(CONFIG, sort_keys=True, indent=2) + "\n"
PROTOCOL_SHA256 = hashlib.sha256(PROTOCOL_JSON.encode("utf-8")).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_int64(values):
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


def write_csv(path, rows):
    assert rows
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


try:
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse
except ModuleNotFoundError:
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyg_lib", "torch_scatter", "torch_sparse", "-f", wheel_url])
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse

from scipy.stats import binomtest, t as student_t, wilcoxon
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

data_candidates = [p for p in Path("/kaggle/input").rglob("data.pt")
                   if "phish/processed/data.pt" in str(p).replace("\\", "/").lower()]
assert len(data_candidates) == 1, [str(p) for p in data_candidates]
data_sha256 = sha256_file(data_candidates[0])
assert data_sha256 == SOURCE_DATA_SHA256
loaded = torch.load(data_candidates[0], map_location="cpu", weights_only=False)
data = loaded[0] if isinstance(loaded, tuple) and len(loaded) == 2 else loaded
assert isinstance(data, HeteroData)
del loaded
gc.collect()
assert set(data.node_types) == {"EOA", "CA"}
assert data["EOA"].x.shape[1] == data["CA"].x.shape[1] == 14
paper_masks = {"train": data["EOA"].train_mask.clone(), "validation": data["EOA"].val_mask.clone(), "test": data["EOA"].test_mask.clone()}
assert tuple(int(m.sum()) for m in paper_masks.values()) == (1657, 553, 553)
labeled_mask = (data["EOA"].y == 0) | (data["EOA"].y == 1)
labeled_ids = torch.where(labeled_mask)[0].cpu().numpy().astype(np.int64)
labeled_y = data["EOA"].y[labeled_mask].cpu().numpy().astype(np.int64)
assert len(labeled_ids) == 2763 and int(labeled_y.sum()) == 1206
fanout_by_relation = {edge_type: [100, 100] for edge_type in data.edge_types}

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("/kaggle/working") / f"rq1_matched_{MODEL_KEY}_A_B_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "protocol.json").write_text(PROTOCOL_JSON, encoding="utf-8")
assert sha256_file(run_dir / "protocol.json") == PROTOCOL_SHA256
print("Model pair:", VARIANT_LABEL)
print("Protocol SHA256:", PROTOCOL_SHA256)
print("Code SHA256:", CODE_SHA256)
print("Data SHA256:", data_sha256)
print("Torch/PyG:", torch.__version__, torch_geometric.__version__)
print("Output:", run_dir)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 77.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 99.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 103.1 MB/s eta 0:00:00
Model pair: {'base': 'LR', 'plus': 'LR+'}
Protocol SHA256: bdc29874c0ed7315fa89a46ae44a94e588a43c1003b4ae51755bf1c7b1d45a35
Code SHA256: 327f9119b367313e20739c28bed4b782af947df7e109051f9da634a3e6eec434
Data SHA256: 8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8
Torch/PyG: 2.10.0+cu128 2.8.0.post1
Output: /kaggle/working/rq1_matched_logistic_regression_A_B_20260925T165448399846Z


## Hàm mô hình, đặc trưng và matched split


In [2]:

import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if MODEL_KEY == "lightgbm":
    try:
        from lightgbm import LGBMClassifier
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
        from lightgbm import LGBMClassifier


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def compute_ufe_by_node_type(batch):
    result = {nt: batch[nt].x.new_zeros((batch[nt].num_nodes, EDGE_STATS_DIM)) for nt in ("EOA", "CA")}
    for edge_type in batch.edge_types:
        source_type, relation, destination_type = edge_type
        store = batch[edge_type]
        if store.edge_index.numel() == 0:
            continue
        attributes = torch.nan_to_num(store.edge_attr[:, :2].to(dtype=batch[source_type].x.dtype), nan=0.0, posinf=0.0, neginf=0.0).clamp_min_(0.0)
        src, dst = store.edge_index[0], store.edge_index[1]
        ones = torch.ones(src.numel(), dtype=attributes.dtype)
        col = RELATION_TO_INDEX[relation] * 6
        result[destination_type][:, col + 0].index_add_(0, dst, ones)
        result[source_type][:, col + 1].index_add_(0, src, ones)
        result[destination_type][:, col + 2].index_add_(0, dst, attributes[:, 0])
        result[source_type][:, col + 3].index_add_(0, src, attributes[:, 0])
        result[destination_type][:, col + 4].index_add_(0, dst, attributes[:, 1])
        result[source_type][:, col + 5].index_add_(0, src, attributes[:, 1])
    return {nt: torch.log1p(v) for nt, v in result.items()}


def materialize_features(mask, sampler_seed, expected, shuffle):
    set_seed(sampler_seed)
    loader = NeighborLoader(data, input_nodes=("EOA", mask), num_neighbors=fanout_by_relation,
                            batch_size=512, shuffle=shuffle, num_workers=0)
    ids_parts, full_parts, label_parts = [], [], []
    started = time.perf_counter()
    for batch in loader:
        n = int(batch["EOA"].batch_size)
        ids = batch["EOA"].n_id[:n].clone(); base = batch["EOA"].x[:n].clone(); labels = batch["EOA"].y[:n].clone()
        assert bool(torch.all((labels == 0) | (labels == 1)))
        batch["EOA"].y[:] = -1
        if "CA" in batch.node_types and hasattr(batch["CA"], "y"): batch["CA"].y[:] = -1
        ufe = compute_ufe_by_node_type(batch)["EOA"][:n]
        full = torch.cat([base, ufe], dim=1)
        assert full.shape == (n, 26) and bool(torch.isfinite(full).all())
        ids_parts.append(ids.cpu()); full_parts.append(full.cpu()); label_parts.append(labels.cpu())
    ids = torch.cat(ids_parts).numpy().astype(np.int64, copy=False)
    full = torch.cat(full_parts).numpy().astype(np.float64, copy=True)
    labels = torch.cat(label_parts).numpy().astype(np.int64, copy=False)
    assert len(ids) == len(set(ids.tolist())) == expected
    assert set(ids.tolist()) == set(torch.where(mask)[0].tolist())
    return ids, full, labels, time.perf_counter() - started


def build_model(seed):
    if MODEL_KEY == "logistic_regression":
        return Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(
            C=SHARED_CONFIG["C"], class_weight=SHARED_CONFIG["class_weight"], solver="liblinear", max_iter=5000, random_state=seed))])
    if MODEL_KEY == "random_forest":
        return RandomForestClassifier(n_estimators=SHARED_CONFIG["n_estimators"], max_depth=SHARED_CONFIG["max_depth"],
                                      min_samples_leaf=1, class_weight=None, random_state=seed, n_jobs=-1)
    if MODEL_KEY == "lightgbm":
        return LGBMClassifier(objective="binary", n_estimators=SHARED_CONFIG["n_estimators"], learning_rate=SHARED_CONFIG["learning_rate"],
                              random_state=seed, n_jobs=-1, verbosity=-1)
    raise ValueError(MODEL_KEY)


def positive_scores(model, x):
    p = model.predict_proba(x); return p[:, int(np.where(model.classes_ == 1)[0][0])]


def metric_bundle(labels, scores, seconds=None):
    labels = np.asarray(labels, dtype=np.int64); scores = np.asarray(scores, dtype=np.float64); pred = (scores >= THRESHOLD).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0, 1]).ravel()
    out = {"pr_auc": float(average_precision_score(labels, scores)), "roc_auc": float(roc_auc_score(labels, scores)),
           "precision": float(precision_score(labels, pred, zero_division=0)), "recall": float(recall_score(labels, pred, zero_division=0)),
           "binary_f1": float(f1_score(labels, pred, zero_division=0)), "micro_f1": float(f1_score(labels, pred, average="micro")),
           "macro_f1": float(f1_score(labels, pred, average="macro")), "mcc": float(matthews_corrcoef(labels, pred)),
           "fpr": float(fp/(fp+tn)), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    if seconds is not None: out["seconds"] = float(seconds)
    return out


def run_matched_split(split_key, masks, sampler_seeds):
    folder = run_dir / split_key; folder.mkdir(parents=True, exist_ok=False)
    train_ids, train_full, y_train, train_ufe_seconds = materialize_features(masks["train"], sampler_seeds["train"], 1657, True)
    val_ids, val_full, y_val, val_ufe_seconds = materialize_features(masks["validation"], sampler_seeds["validation"], 553, True)
    assert set(train_ids).isdisjoint(val_ids)
    rows, paths = [], {}
    for variant in VARIANTS:
        x_train = train_full[:, :14] if variant == "base" else train_full
        x_val = val_full[:, :14] if variant == "base" else val_full
        for seed in MODEL_SEEDS:
            set_seed(seed); model = build_model(seed)
            started = time.perf_counter(); model.fit(x_train, y_train); fit_seconds = time.perf_counter() - started
            val_started = time.perf_counter(); val_scores = positive_scores(model, x_val); val_seconds = time.perf_counter() - val_started
            vm = metric_bundle(y_val, val_scores, val_seconds)
            path = folder / f"{variant}_seed_{seed}.joblib"; joblib.dump(model, path); paths[(variant, seed)] = path
            rows.append({"variant": variant, "seed": seed, "fit_seconds": fit_seconds,
                         **{f"validation_{k}": v for k, v in vm.items()}, "checkpoint_sha256": sha256_file(path)})
    write_csv(folder / "validation_runs.csv", rows)
    decision = {"status": "FROZEN VALIDATION DECISION (fixed shared config)", "split_key": split_key,
                "protocol_sha256": PROTOCOL_SHA256, "code_sha256": CODE_SHA256, "source_data_sha256": data_sha256,
                "shared_config": SHARED_CONFIG, "sampler_seeds": sampler_seeds,
                "train_target_ids_sha256": sha256_int64(sorted(train_ids)), "validation_target_ids_sha256": sha256_int64(sorted(val_ids)),
                "train_features_sha256": hashlib.sha256(train_full.tobytes()).hexdigest(),
                "validation_features_sha256": hashlib.sha256(val_full.tobytes()).hexdigest(),
                "checkpoint_sha256": {f"{r['variant']}_seed_{r['seed']}": r["checkpoint_sha256"] for r in rows},
                "test_features_or_labels_loaded_before_freeze": False, "test_metrics_computed_before_freeze": False}
    (folder / "frozen_validation_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")

    test_ids, test_full, y_test, test_ufe_seconds = materialize_features(masks["test"], sampler_seeds["test"], 553, False)
    assert set(test_ids).isdisjoint(train_ids) and set(test_ids).isdisjoint(val_ids)
    results, predictions, cube = [], [], {}
    for variant in VARIANTS:
        x_test = test_full[:, :14] if variant == "base" else test_full
        scores_by_seed = []
        for seed in MODEL_SEEDS:
            path = paths[(variant, seed)]; assert sha256_file(path) == decision["checkpoint_sha256"][f"{variant}_seed_{seed}"]
            model = joblib.load(path); started = time.perf_counter(); scores = positive_scores(model, x_test); seconds = time.perf_counter()-started
            tm = metric_bundle(y_test, scores, seconds); scores_by_seed.append(scores)
            vr = next(r for r in rows if r["variant"] == variant and r["seed"] == seed)
            results.append({"split_key": split_key, "variant": variant, "model": VARIANT_LABEL[variant], "seed": seed,
                            "validation_macro_f1": vr["validation_macro_f1"], "fit_seconds": vr["fit_seconds"],
                            **{f"test_{k}": v for k, v in tm.items()}})
            predictions.extend({"split_key": split_key, "variant": variant, "model": VARIANT_LABEL[variant], "seed": seed,
                                "node_id": int(i), "label": int(y), "score": float(s), "prediction": int(s >= THRESHOLD), "threshold": THRESHOLD}
                               for i, y, s in zip(test_ids, y_test, scores))
        cube[variant] = np.asarray(scores_by_seed)
    write_csv(folder / "test_results.csv", results); write_csv(folder / "test_predictions.csv", predictions)
    split_result = {"split_key": split_key, "protocol_sha256": PROTOCOL_SHA256, "code_sha256": CODE_SHA256,
                    "source_data_sha256": data_sha256, "test_target_ids_sorted_sha256": sha256_int64(sorted(test_ids)),
                    "test_features_sha256": hashlib.sha256(test_full.tobytes()).hexdigest(),
                    "base_mean_macro_f1": float(np.mean([r["test_macro_f1"] for r in results if r["variant"]=="base"])),
                    "plus_mean_macro_f1": float(np.mean([r["test_macro_f1"] for r in results if r["variant"]=="plus"])),
                    "ufe_seconds": {"train": train_ufe_seconds, "validation": val_ufe_seconds, "test": test_ufe_seconds}, "decision": decision}
    split_result["delta"] = split_result["plus_mean_macro_f1"] - split_result["base_mean_macro_f1"]
    np.savez_compressed(folder / "score_cube.npz", base=cube["base"], plus=cube["plus"], labels=y_test, test_ids=test_ids)
    split_result["score_cube_sha256"] = sha256_file(folder / "score_cube.npz")
    (folder / "split_result.json").write_text(json.dumps(split_result, indent=2), encoding="utf-8")
    return split_result, cube, y_test


def mean_macro(score_matrix, labels, sample):
    truth = labels[sample].astype(bool)
    pred = score_matrix[:, sample] >= THRESHOLD
    tp = (pred & truth).sum(1); fp = (pred & ~truth).sum(1)
    fn = (~pred & truth).sum(1); tn = (~pred & ~truth).sum(1)
    f1_pos = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(len(tp)), where=(2 * tp + fp + fn) > 0)
    f1_neg = np.divide(2 * tn, 2 * tn + fp + fn, out=np.zeros(len(tn)), where=(2 * tn + fp + fn) > 0)
    return float(np.mean((f1_pos + f1_neg) / 2))


def summarize_table_a(result_a, cube_a, labels_a):
    nodes = np.arange(len(labels_a))
    observed = mean_macro(cube_a["plus"], labels_a, nodes) - mean_macro(cube_a["base"], labels_a, nodes)
    rng = np.random.default_rng(20260925)
    boot = np.empty(BOOTSTRAP_REPETITIONS)
    for i in range(BOOTSTRAP_REPETITIONS):
        sample = rng.integers(0, len(labels_a), size=len(labels_a))
        boot[i] = mean_macro(cube_a["plus"], labels_a, sample) - mean_macro(cube_a["base"], labels_a, sample)
    ci = np.quantile(boot, [0.025, 0.975])
    p = min(1.0, 2 * min((np.sum(boot <= 0) + 1) / (BOOTSTRAP_REPETITIONS + 1),
                         (np.sum(boot >= 0) + 1) / (BOOTSTRAP_REPETITIONS + 1)))
    per_seed = []
    for k, seed in enumerate(MODEL_SEEDS):
        plus_ok = (cube_a["plus"][k] >= THRESHOLD) == labels_a
        base_ok = (cube_a["base"][k] >= THRESHOLD) == labels_a
        only_plus = int(np.sum(plus_ok & ~base_ok)); only_base = int(np.sum(~plus_ok & base_ok))
        per_seed.append({"seed": seed,
                         "plus_macro_f1": mean_macro(cube_a["plus"][[k]], labels_a, nodes),
                         "base_macro_f1": mean_macro(cube_a["base"][[k]], labels_a, nodes),
                         "only_plus_correct": only_plus, "only_base_correct": only_base,
                         "mcnemar_p": 1.0 if only_plus + only_base == 0 else float(binomtest(min(only_plus, only_base), only_plus + only_base, 0.5).pvalue)})
    return {"role": "retrospective matched comparison on the previously viewed paper split",
            "base_mean_macro_f1": result_a["base_mean_macro_f1"], "plus_mean_macro_f1": result_a["plus_mean_macro_f1"],
            "delta": observed, "bootstrap_95ci": [float(ci[0]), float(ci[1])], "bootstrap_p_two_sided": float(p),
            "seeds_plus_better": int(sum(r["plus_macro_f1"] > r["base_macro_f1"] for r in per_seed)), "per_seed": per_seed}


def summarize_table_b(split_results):
    deltas = np.asarray([r["delta"] for r in split_results])
    k = len(deltas); assert k == 10
    ratio = EXPECTED_COUNTS["test"] / EXPECTED_COUNTS["train"]
    variance = float(deltas.var(ddof=1))
    se = float(np.sqrt((1 / k + ratio) * variance))
    mean_delta = float(deltas.mean())
    if se == 0:
        t_stat, p, ci = (float("inf") if mean_delta else 0.0), (0.0 if mean_delta else 1.0), (mean_delta, mean_delta)
    else:
        t_stat = mean_delta / se
        p = float(2 * student_t.sf(abs(t_stat), df=k - 1))
        crit = float(student_t.ppf(0.975, df=k - 1))
        ci = (mean_delta - crit * se, mean_delta + crit * se)
    try:
        wp = float(wilcoxon(deltas).pvalue)
    except ValueError:
        wp = 1.0
    return {
        "role": "pre-registered re-split stability; not an external holdout; paper split excluded",
        "per_split": [{"split_key": r["split_key"], "base": r["base_mean_macro_f1"], "plus": r["plus_mean_macro_f1"], "delta": r["delta"]} for r in split_results],
        "base_mean_over_splits": float(np.mean([r["base_mean_macro_f1"] for r in split_results])),
        "plus_mean_over_splits": float(np.mean([r["plus_mean_macro_f1"] for r in split_results])),
        "mean_delta": mean_delta, "sd_delta": float(np.sqrt(variance)),
        "nadeau_bengio": {"se": se, "t": t_stat, "df": 9, "p_two_sided_raw": p, "ci95": [ci[0], ci[1]], "ratio_test_train": ratio},
        "descriptive_wilcoxon_p": wp, "splits_plus_better": int(np.sum(deltas > 0)), "splits_tied": int(np.sum(deltas == 0)),
        "raw_pair_rule_met": bool(mean_delta >= MINIMUM_PRACTICAL_DELTA and ci[0] > 0 and p < ALPHA),
        "global_holm_pending": True,
    }


## Chạy Bảng A rồi 10 split Bảng B


In [3]:

result_a,cube_a,labels_a=run_matched_split("A_paper_split",paper_masks,PAPER_SAMPLER_SEEDS)
table_a=summarize_table_a(result_a,cube_a,labels_a)
(run_dir/"table_a_summary.json").write_text(json.dumps(table_a,indent=2),encoding="utf-8")
print("TABLE A",VARIANT_LABEL["base"],table_a["base_mean_macro_f1"],VARIANT_LABEL["plus"],table_a["plus_mean_macro_f1"],"delta",table_a["delta"])

split_results=[]
for split_seed in RESPLIT_SEEDS:
    pos=np.arange(len(labeled_ids));trainval_pos,test_pos=train_test_split(pos,test_size=.20,random_state=split_seed,stratify=labeled_y)
    train_pos,val_pos=train_test_split(trainval_pos,test_size=.25,random_state=split_seed,stratify=labeled_y[trainval_pos])
    masks={}
    for name,p in (("train",train_pos),("validation",val_pos),("test",test_pos)):
        m=torch.zeros(data["EOA"].num_nodes,dtype=torch.bool);m[torch.from_numpy(labeled_ids[p])]=True;masks[name]=m
    seeds={"train":split_seed*10+1,"validation":split_seed*10+2,"test":split_seed*10+3}
    result,_,_=run_matched_split(f"B_split_{split_seed}",masks,seeds);split_results.append(result)
    print(result["split_key"],result["base_mean_macro_f1"],result["plus_mean_macro_f1"],result["delta"],flush=True)

table_b=summarize_table_b(split_results)
write_csv(run_dir/"table_b_per_split.csv",table_b["per_split"])
final={"status":"completed matched pair Table A + Table B","model_key":MODEL_KEY,"model_label":MODEL_LABEL,"variant_labels":VARIANT_LABEL,"protocol_sha256":PROTOCOL_SHA256,"code_sha256":CODE_SHA256,"protocol":CONFIG,"source_data_sha256":data_sha256,"table_a":table_a,"table_b":table_b,"post_test_selection":False,"protocol_changed_after_results":False}
(run_dir/"final_summary.json").write_text(json.dumps(final,indent=2),encoding="utf-8")
shutil.make_archive(str(run_dir),"zip",run_dir)
print("TABLE B mean:",table_b["base_mean_over_splits"],"->",table_b["plus_mean_over_splits"],"delta",table_b["mean_delta"])
print("NB:",table_b["nadeau_bengio"])
print("Raw pair rule met:",table_b["raw_pair_rule_met"],"| final Holm pending across all six pairs")
print("Output:",run_dir,"ZIP:",str(run_dir)+".zip")


TABLE A LR 0.9147581737502009 LR+ 0.9761309494164717 delta 0.06137277566627086
B_split_101 0.9075714524486044 0.9687866261600014 0.061215173711397064
B_split_202 0.8787855986128605 0.9669942702530638 0.0882086716402033
B_split_303 0.9003848358306723 0.9723806574702081 0.07199582163953577
B_split_404 0.9002868617326449 0.9687579966036244 0.06847113487097944
B_split_505 0.9147581737502009 0.9596596636426336 0.0449014898924327
B_split_606 0.9183522811161968 0.9779570035343201 0.05960472241812331
B_split_707 0.9291409984722291 0.9559538032656312 0.02681280479340209
B_split_808 0.8949935831958304 0.9651755144789916 0.07018193128316119
B_split_909 0.9020018377526909 0.9650494802494801 0.06304764249678918
B_split_1010 0.9093989199958058 0.9725521079782535 0.06315318798244773
TABLE B mean: 0.9055674542907735 -> 0.9673267123636208 delta 0.06175925807284718
NB: {'se': 0.010831309988280928, 't': 5.701919540634363, 'df': 9, 'p_two_sided_raw': 0.0002935938800758618, 'ci95': [0.03725713259976432, 0.